# nb_lib_delta
Delta Lake helpers shared by all layers. Requires `%run nb_config` and `%run nb_lib_logging` first.

| Function | Purpose |
|---|---|
| `table_exists`, `read_table` | catalog helpers |
| `append_table`, `overwrite_table` | writes (overwrite supports `replace_where` for partial refresh) |
| `merge_upsert` | MERGE on business keys; creates the table on first run |
| `dedupe_latest` | keep the newest row per key (pure DataFrame function) |
| `get_watermark`, `set_watermark` | incremental-load bookmarks in `ops.watermark` |

In [ ]:
from delta.tables import DeltaTable
from pyspark.sql import DataFrame, Window
from pyspark.sql import functions as F

_delta_log = get_logger("lib_delta")


def table_exists(name: str) -> bool:
    return spark.catalog.tableExists(name)


def read_table(name: str) -> DataFrame:
    return spark.read.table(name)


def append_table(df: DataFrame, name: str, merge_schema: bool = False):
    (df.write.format("delta").mode("append")
       .option("mergeSchema", str(merge_schema).lower())
       .saveAsTable(name))


def overwrite_table(df: DataFrame, name: str, replace_where: str = None):
    """Full overwrite, or partial overwrite of rows matching `replace_where`
    (e.g. "date_key IN (20261101, 20261102)")."""
    writer = df.write.format("delta").mode("overwrite")
    if replace_where:
        writer = writer.option("replaceWhere", replace_where)
    writer.saveAsTable(name)

In [ ]:
def dedupe_latest(df: DataFrame, keys: list, order_by: list) -> DataFrame:
    """Keep exactly one row per key: the one with the greatest `order_by` values.

    Nulls in order_by sort last, so a row with a valid timestamp always beats one without.
    Exact duplicates collapse to one row.
    """
    w = Window.partitionBy(*keys).orderBy(*[F.col(c).desc_nulls_last() for c in order_by])
    return df.withColumn("_rn", F.row_number().over(w)).filter("_rn = 1").drop("_rn")


def merge_upsert(df: DataFrame, name: str, keys: list, update_condition: str = None) -> dict:
    """Upsert `df` into table `name` matching on `keys`.

    * First run: creates the table from df.
    * update_condition: optional SQL on aliases t (target) and s (source), e.g.
      "s.updated_at > t.updated_at" so late, older records never overwrite newer ones.
    * Source must already be unique per key (use dedupe_latest), or MERGE fails.
    Returns {"inserted": n, "updated": n, "source_rows": n}.
    """
    if not table_exists(name):
        df.write.format("delta").saveAsTable(name)
        n = spark.read.table(name).count()
        _delta_log.info("created %s with %d rows", name, n)
        return {"inserted": n, "updated": 0, "source_rows": n}

    on = " AND ".join(f"t.`{k}` = s.`{k}`" for k in keys)
    (DeltaTable.forName(spark, name).alias("t")
        .merge(df.alias("s"), on)
        .whenMatchedUpdateAll(condition=update_condition)
        .whenNotMatchedInsertAll()
        .execute())

    m = (DeltaTable.forName(spark, name).history(1)
         .select("operationMetrics").first()[0]) or {}
    result = {
        "inserted": int(m.get("numTargetRowsInserted", 0)),
        "updated": int(m.get("numTargetRowsUpdated", 0)),
        "source_rows": int(m.get("numSourceRows", 0)),
    }
    _delta_log.info("merged into %s: %s", name, result)
    return result

In [ ]:
def get_watermark(pipeline: str, source: str, default=None, table: str = None):
    """Last successfully processed value for (pipeline, source), or `default`."""
    table = table or tbl("ops", "watermark")
    if not table_exists(table):
        return default
    row = (spark.read.table(table)
           .filter((F.col("pipeline") == pipeline) & (F.col("source") == source))
           .select("watermark_value").first())
    return row[0] if row else default


def set_watermark(pipeline: str, source: str, value, table: str = None):
    """Store a new watermark. Call only AFTER the data write succeeded."""
    table = table or tbl("ops", "watermark")
    df = spark.createDataFrame([(pipeline, source, str(value), utc_now())], OPS_TABLES["watermark"])
    merge_upsert(df, table, ["pipeline", "source"])
    _delta_log.info("watermark %s/%s -> %s", pipeline, source, value)


print("[nb_lib_delta] loaded")